# Sasquatch health check
Check Sasquatch /health endpoints 

In [ ]:
import os
from httpx import AsyncClient
from lsst.rsp import RSPDiscovery, RSPInternalDiscovery, get_influxdb_location, list_influxdb_labels
from urllib.parse import urljoin

Create an RSP client to talk to Sasquatch APIs

In [ ]:
client = AsyncClient()

Get token information.

In [ ]:
discovery = RSPInternalDiscovery()
session = discovery.get_session()
gafaelfawr_url = discovery.get_internal_service_url("gafaelfawr", version="v1")
r = session.get(gafaelfawr_url + "/token-info")
token_info = r.json()
print(f"Authenticated user: {token_info['username']}")

Find an InfluxDB database that can be queried.

In [ ]:
influxdb_labels = list_influxdb_labels(local=True)
print(f"Available InfluxDB labels: {', '.join(influxdb_labels)}")
influxdb_location = None
if influxdb_labels:
    influxdb_location = get_influxdb_location(influxdb_labels[0])
    print(f"URL of first InfluxDB database: {influxdb_location.url}")

## Check InfluxDB health 

In [ ]:
if influxdb_location:
    r = await client.get(influxdb_location.url + "health")
    assert r.status_code == 200
    version = r.json()["version"]
    message = r.json()["message"]
    print(f"InfluxDB {version} is {message}")

## Test InfluxDB connection credentials

Use Repertoire to retrieve InfluxDB connection credentials 

## Test InfluxDB queries

Query lsst.square.metrics database

## Check Chronograf health

This is not a completely correct way to get the Chronograf URL since it assumes it is relative to the InfluxDB URL of the first available label. It should either be returned as part of the InfluxDB location information or we should expose UI services to Nublado service discovery. Or, alternately, we can just not probe the health of UI services.

In [ ]:
if influxdb_location:
    chronograf_url = urljoin(influxdb_location.url, "/chronograf")
    r = await client.get(chronograf_url)
    assert r.status_code == 200

## Check Kafdrop health

Requires token with exec:internal-tools scope

In [ ]:
assert "exec:internal-tools" in token_info['scopes']

In [ ]:
if influxdb_location:
    token = RSPDiscovery.get_token()
    kafdrop_url = urljoin(influxdb_location.url, "/kafdrop")
    r = await client.get(kafdrop_url + "/actuator/health", headers={"Authorization": f"Bearer {token}"})
    assert r.status_code == 200
    status = r.json()['status']
    print(f"Kafdrop is {status}")

## Check Schema Registry health

Requires token with write:sasquatch scope 

In [ ]:
assert "write:sasquatch" in token_info['scopes']

In [ ]:
if influxdb_location:
    r = await client.get(influxdb_location.schema_registry, headers={"Authorization": f"Bearer {token}"})
    r.status_code == 200